# Feast Feature Store Integration - Complete Notebook

## Assignment: Integrating Feast Feature Store into the IRIS Pipeline

This notebook walks through all tasks step by step:
- **TASK 1-2**: Feature repository initialization and definitions ✅ DONE
- **TASK 3**: Materialize features to online store
- **TASK 4**: Fetch features for training (offline store)
- **TASK 5**: Fetch features for inference (online store)
- **TASK 6**: GCP BigQuery backend (optional)

### Key Concept: Eliminating Training-Serving Skew
By using Feast, both training and inference use the **same features** from the **same source**, eliminating data inconsistencies.

## Setup: Install Dependencies

Run this cell first to install all required packages.

In [ ]:
# Install required packages
import subprocess
import sys

packages = [
    'feast==0.31.1',
    'pandas==2.0.3',
    'scikit-learn==1.3.0',
    'joblib==1.3.1',
    'numpy==1.24.3',
]

print("Installing required packages...")
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

print("\n✓ All packages installed successfully!")

## Import Libraries

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import warnings

# ML Libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

# Feast
from feast import FeatureStore

warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully!")
print(f"\nPython version: {sys.version}")
print(f"Current directory: {os.getcwd()}")

## Setup: Configure Paths

Set up the directory structure for the project.

In [ ]:
# Define paths
PROJECT_ROOT = Path.cwd()
FEATURE_REPO = PROJECT_ROOT / "feature_repo"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
MODELS_DIR = PROJECT_ROOT / "models"
DATA_FILE = PROJECT_ROOT / "iris_data_adapted_for_feast.csv"

# Create directories if they don't exist
MODELS_DIR.mkdir(exist_ok=True)

print("Project Structure:")
print(f"  Project Root: {PROJECT_ROOT}")
print(f"  Feature Repo: {FEATURE_REPO}")
print(f"  Data File: {DATA_FILE}")
print(f"  Models Dir: {MODELS_DIR}")

# Check if data file exists
if DATA_FILE.exists():
    print(f"\n✓ Data file found: {DATA_FILE}")
else:
    print(f"\n✗ Data file NOT found at {DATA_FILE}")
    print("Please copy iris_data_adapted_for_feast.csv to the project root")

## Step 1: Review the Dataset

Let's examine the iris dataset structure to understand what we're working with.

In [ ]:
# Load and examine the dataset
iris_data = pd.read_csv(str(DATA_FILE))

print("Dataset Overview:")
print(f"  Shape: {iris_data.shape}")
print(f"  Columns: {list(iris_data.columns)}")
print(f"\nFirst 5 rows:")
print(iris_data.head())

print(f"\nData Types:")
print(iris_data.dtypes)

print(f"\nUnique iris IDs: {iris_data['iris_id'].unique()}")
print(f"Unique species: {iris_data['species'].unique()}")
print(f"Date range: {iris_data['event_timestamp'].min()} to {iris_data['event_timestamp'].max()}")

## TASK 1 & 2: Review Feature Repository Setup ✅ DONE

The feature repository and feature definitions are already set up. Let's review them.

In [ ]:
# Display the feature_store.yaml configuration
print("Feature Store Configuration (feature_store.yaml):")
print("="*70)

yaml_path = FEATURE_REPO / "feature_store.yaml"
with open(yaml_path, 'r') as f:
    print(f.read())

print("\n" + "="*70)

In [ ]:
# Display the features.py definitions
print("Feature Definitions (features.py):")
print("="*70)

features_path = FEATURE_REPO / "features.py"
with open(features_path, 'r') as f:
    print(f.read())

print("\n" + "="*70)

## TASK 3: Materialize Features to Online Store

**What's happening:**
- Features are read from the **offline store** (CSV file)
- They are copied to the **online store** (SQLite database)
- This makes them available for fast real-time inference

**Why:** Materialization prepares features for production serving where low-latency access is needed.

In [ ]:
print("="*70)
print("TASK 3: MATERIALIZING FEATURES TO ONLINE STORE")
print("="*70)

try:
    # Load the feature store
    fs = FeatureStore(repo_path=str(FEATURE_REPO))
    print("✓ Feast Feature Store loaded successfully")
    
    # Materialize features
    start_date = datetime.now() - timedelta(days=60)
    end_date = datetime.now()
    
    print(f"\nMaterializing features from {start_date.date()} to {end_date.date()}...")
    fs.materialize(start_date=start_date, end_date=end_date)
    
    print("\n" + "="*70)
    print("✓ MATERIALIZATION SUCCESSFUL!")
    print("="*70)
    print("\nYour online store is now populated with features for inference.")
    print("You can now use fs.get_online_features() for real-time predictions.")
    
except Exception as e:
    print(f"\n✗ ERROR during materialization: {str(e)}")
    print("\nTroubleshooting:")
    print("1. Ensure CSV path is correct")
    print("2. Verify CSV has 'event_timestamp' and 'iris_id' columns")
    import traceback
    traceback.print_exc()

## TASK 4: Train Model Using Features from Offline Store

**What's happening:**
1. We get entity IDs and timestamps from the iris data
2. We call `fs.get_historical_features()` to fetch features from the **offline store** (CSV)
3. We train a model using these features
4. We save the trained model

**Key Point:** The model is trained using features fetched from Feast, not directly from the CSV. This ensures consistency with inference!

In [ ]:
print("="*70)
print("TASK 4: TRAINING WITH FEAST OFFLINE STORE")
print("="*70)

try:
    # 1. LOAD THE FEATURE STORE
    fs = FeatureStore(repo_path=str(FEATURE_REPO))
    print("✓ Feast Feature Store loaded")
    
    # 2. LOAD IRIS DATA TO GET ENTITY KEYS
    iris_data = pd.read_csv(str(DATA_FILE))
    print(f"✓ Loaded iris data: {iris_data.shape[0]} rows, {iris_data.shape[1]} columns")
    
    # 3. CREATE ENTITY DATAFRAME
    entity_df = iris_data[["iris_id", "event_timestamp"]].copy()
    print(f"✓ Created entity DataFrame for feature lookup")
    print(f"\n  Sample entities (first 5):")
    print(entity_df.head())
    
    # 4. FETCH HISTORICAL FEATURES FROM OFFLINE STORE
    print("\n▶ Fetching historical features from Feast offline store...")
    print("  (This reads from your CSV file)")
    
    training_features = fs.get_historical_features(
        entity_df=entity_df,
        features=[
            "iris_measurements:sepal_length",
            "iris_measurements:sepal_width",
            "iris_measurements:petal_length",
            "iris_measurements:petal_width",
            "iris_measurements:species",
        ],
    ).to_df()
    
    print(f"✓ Features fetched successfully!")
    print(f"  Shape: {training_features.shape}")
    print(f"\n  Sample features (first 5):")
    print(training_features.head())
    
    # 5. PREPARE DATA FOR TRAINING
    feature_columns = [
        "sepal_length", "sepal_width", "petal_length", "petal_width"
    ]
    X = training_features[feature_columns].values
    
    # Convert species string to numeric
    species_map = {"setosa": 0, "versicolor": 1, "virginica": 2}
    y = training_features["species"].map(species_map).values
    
    print(f"\n▶ Preparing training data:")
    print(f"  X shape: {X.shape}")
    print(f"  y shape: {y.shape}")
    print(f"  Classes: {species_map}")
    
    # 6. TRAIN THE MODEL
    print(f"\n▶ Training RandomForest classifier...")
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X, y)
    
    # Evaluate
    y_pred = model.predict(X)
    accuracy = accuracy_score(y, y_pred)
    print(f"✓ Model trained!")
    print(f"  Training accuracy: {accuracy:.2%}")
    
    print(f"\n  Classification Report:")
    print(classification_report(y, y_pred, target_names=["setosa", "versicolor", "virginica"]))
    
    # 7. SAVE THE MODEL
    model_path = MODELS_DIR / "iris_model_feast.joblib"
    joblib.dump(model, str(model_path))
    
    print("="*70)
    print(f"✓ MODEL TRAINING COMPLETE!")
    print("="*70)
    print(f"Model saved to: {model_path}")
    print(f"\nKey Achievement:")
    print("  Your model was trained using features from Feast,")
    print("  not directly from the CSV. This ensures consistency")
    print("  with how inference will retrieve features!")

except Exception as e:
    print(f"\n✗ ERROR during training: {str(e)}")
    print("\nTroubleshooting:")
    print("1. Did you run TASK 3 (Materialize) first?")
    print("2. Check that iris_data_adapted_for_feast.csv exists")
    print("3. Verify features.py is correctly defined")
    import traceback
    traceback.print_exc()

## TASK 5: Perform Inference Using Features from Online Store

**What's happening:**
1. We load the trained model
2. We specify iris IDs to predict for
3. We call `fs.get_online_features()` to fetch features from the **online store** (SQLite)
4. We make predictions using the fetched features

**Key Difference from Training:**
- Training used `get_historical_features()` (offline store, batch retrieval)
- Inference uses `get_online_features()` (online store, real-time lookup)
- **Both use the SAME features!** This eliminates training-serving skew.

In [ ]:
print("="*70)
print("TASK 5: INFERENCE WITH FEAST ONLINE STORE")
print("="*70)

try:
    # 1. LOAD FEATURE STORE AND MODEL
    fs = FeatureStore(repo_path=str(FEATURE_REPO))
    print("✓ Feast Feature Store loaded")
    
    model_path = MODELS_DIR / "iris_model_feast.joblib"
    if not model_path.exists():
        print(f"✗ Model not found at {model_path}")
        print("  Please run TASK 4 (Training) first")
        raise FileNotFoundError(f"Model not found: {model_path}")
    
    model = joblib.load(str(model_path))
    print(f"✓ Model loaded from {model_path}")
    
    # 2. SELECT IRIS PLANTS FOR INFERENCE
    iris_ids_to_predict = [1001, 1002, 1003]
    print(f"\n▶ Predicting for iris IDs: {iris_ids_to_predict}")
    
    # 3. CREATE ENTITY DATAFRAME FOR ONLINE STORE LOOKUP
    entity_df = pd.DataFrame({
        "iris_id": iris_ids_to_predict,
    })
    print(f"✓ Created entity DataFrame for online lookup")
    
    # 4. FETCH FEATURES FROM ONLINE STORE (REAL-TIME)
    print(f"\n▶ Fetching features from Feast ONLINE STORE...")
    print(f"  (This is fast, low-latency lookup for production)")
    
    online_features = fs.get_online_features(
        entity_rows=entity_df.to_dict(orient="records"),
        features=[
            "iris_measurements:sepal_length",
            "iris_measurements:sepal_width",
            "iris_measurements:petal_length",
            "iris_measurements:petal_width",
        ],
    ).to_df()
    
    print(f"✓ Features fetched from online store!")
    print(f"\n  Features:")
    print(online_features)
    
    # 5. PREPARE DATA FOR INFERENCE
    feature_columns = [
        "sepal_length", "sepal_width", "petal_length", "petal_width"
    ]
    X_inference = online_features[feature_columns].values
    
    # 6. MAKE PREDICTIONS
    print(f"\n▶ Making predictions...")
    predictions = model.predict(X_inference)
    probabilities = model.predict_proba(X_inference)
    
    # 7. DISPLAY RESULTS
    species_names = {0: "setosa", 1: "versicolor", 2: "virginica"}
    
    print("\n" + "="*70)
    print("INFERENCE RESULTS")
    print("="*70)
    
    for i, iris_id in enumerate(iris_ids_to_predict):
        pred = predictions[i]
        probs = probabilities[i]
        species = species_names[pred]
        confidence = probs[pred]
        
        print(f"\n🌸 Iris ID: {iris_id}")
        print(f"   Predicted Species: {species}")
        print(f"   Confidence: {confidence:.2%}")
        print(f"   All probabilities:")
        for class_id, prob in enumerate(probs):
            print(f"     - {species_names[class_id]}: {prob:.2%}")
    
    # 8. VERIFY CONSISTENCY
    print("\n" + "="*70)
    print("✓ INFERENCE COMPLETE!")
    print("="*70)
    print("\nKey Achievement:")
    print("  Your inference used the SAME features as training,")
    print("  fetched from the ONLINE store for real-time serving.")
    print("  This eliminates training-serving skew!")
    
except Exception as e:
    print(f"\n✗ ERROR during inference: {str(e)}")
    print("\nTroubleshooting:")
    print("1. Did you run TASK 3 (Materialize) first?")
    print("2. Did you run TASK 4 (Training) first?")
    print("3. Check that models/iris_model_feast.joblib exists")
    import traceback
    traceback.print_exc()

## Summary: What We Accomplished

### The Problem (Training-Serving Skew)
Without Feast:
- Training: Read features from CSV
- Inference: Compute features on-the-fly or read from different source
- **Result:** Different features used → predictions inconsistent with training

### The Solution (Feast)
With Feast:
- **Single Source of Truth:** Define features once in `features.py`
- **Training:** Fetch from offline store using `get_historical_features()`
- **Inference:** Fetch from online store using `get_online_features()`
- **Result:** Same features, same definitions, consistent predictions

### Architecture
```
Raw Data (iris_data_adapted_for_feast.csv)
    ↓
    ├─→ [OFFLINE STORE] ← Training: get_historical_features()
    │
    └─→ Materialize → [ONLINE STORE] ← Inference: get_online_features()
                           ↓
                    SQLite or BigQuery
```

## TASK 6 (OPTIONAL): GCP BigQuery Backend

For production ML systems, you can replace SQLite with Google BigQuery for:
- **Scalability**: Handle unlimited data
- **Team Access**: Share features across teams
- **Production Ready**: Enterprise-grade data warehouse

### Trade-offs
| Aspect | SQLite (Local) | BigQuery (GCP) |
|--------|---|---|
| Setup | Instant | Requires GCP |
| Latency | ~1ms | ~100ms |
| Scalability | Limited | Unlimited |
| Cost | Free | Pay per query |
| Team Access | No | ✓ Yes |
| Production | No | ✓ Yes |

### To Use BigQuery
See the detailed guide in `GCP_BIGQUERY_SETUP.py` for step-by-step instructions.

In [ ]:
# Display GCP setup guide
print("GCP BigQuery Setup Guide")
print("="*70)

gcp_guide_path = PROJECT_ROOT / "GCP_BIGQUERY_SETUP.py"
if gcp_guide_path.exists():
    print(f"See: {gcp_guide_path}")
    print("\nFor production setup with BigQuery:")
    print("1. Create GCP project and BigQuery dataset")
    print("2. Update feature_store.yaml with BigQuery config")
    print("3. Update features.py to use BigQuerySource")
    print("4. Authenticate: gcloud auth application-default login")
    print("5. Run: feast apply (updates registry in Cloud Storage)")
    print("6. Run: feast materialize (copies to BigQuery online store)")
    print("\nNote: Rest of the code stays the same!")
else:
    print("GCP setup guide not found")

## Verification: Check What We Built

Let's verify all the files and outputs are in place.

In [ ]:
import os

print("Project Files Status")
print("="*70)

# Check essential files
files_to_check = {
    "Feature Store Config": FEATURE_REPO / "feature_store.yaml",
    "Feature Definitions": FEATURE_REPO / "features.py",
    "Data File": DATA_FILE,
    "Trained Model": MODELS_DIR / "iris_model_feast.joblib",
    "Online Store DB": FEATURE_REPO / "data" / "online_store.db",
}

for name, path in files_to_check.items():
    if path.exists():
        size = os.path.getsize(path)
        print(f"✓ {name:25} ({size:,} bytes)")
    else:
        print(f"✗ {name:25} [NOT FOUND]")

print("\nProject Root Contents:")
for item in sorted(PROJECT_ROOT.iterdir()):
    if item.name.startswith('.'):
        continue
    if item.is_dir():
        print(f"  📁 {item.name}/")
    else:
        print(f"  📄 {item.name}")

## Complete! 🎉

You have successfully completed the Feast Feature Store integration:

✅ **TASK 1**: Feature repository initialized
✅ **TASK 2**: Entities, data sources, and feature views defined
✅ **TASK 3**: Features materialized to online store
✅ **TASK 4**: Model trained using features from offline store
✅ **TASK 5**: Inference performed using features from online store
✅ **TASK 6**: (Optional) GCP BigQuery backend guide provided

### Key Learnings
- Feast eliminates **training-serving skew** by providing one source of truth
- **Offline store** = historical data for training (batch retrieval)
- **Online store** = materialized data for inference (real-time retrieval)
- **Features** are defined once, used consistently everywhere
- Local SQLite for development, BigQuery for production

### Next Steps for Submission
1. ✅ Run all notebooks cells (done!)
2. 🎥 Screen record running these cells
3. 📝 Document outputs and findings
4. 📤 Submit with code and video

**Great work!** 🚀